# Quiz 2: Dollar Bill Value Detection

**Objective**: Build a CNN model to detect and classify dollar bill values from images

---

## Workflow:
1. Upload and extract dataset ZIP file
2. Create train/test split by moving images
3. Build CNN model for classification
4. Train the model
5. Evaluate on test data
6. Visualize results

## Step 1: Setup and Import Libraries

In [ ]:
# Install required packages
!pip install -q scikit-learn matplotlib seaborn pillow

In [ ]:
# Import libraries
import os
import shutil
import zipfile
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import json

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Scikit-learn
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set seeds for reproducibility
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## Step 2: Mount Google Drive and Upload Dataset

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Option 1: Upload ZIP file directly
from google.colab import files

print("Please upload your dollar bill dataset ZIP file...")
uploaded = files.upload()

# Get the uploaded file name
zip_filename = list(uploaded.keys())[0]
print(f"\nUploaded file: {zip_filename}")

In [ ]:
# Alternative Option 2: If file is already in Google Drive, uncomment and modify path
# zip_filename = '/content/drive/MyDrive/path_to_your_dataset.zip'

## Step 3: Extract Dataset and Explore Structure

In [ ]:
# Extract ZIP file
extract_path = '/content/dollar_dataset'

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Dataset extracted to: {extract_path}")
print("\nDataset structure:")
!ls -R {extract_path}

In [ ]:
# Find the actual data directory (sometimes ZIP has nested folders)
def find_data_root(base_path):
    """Find the root directory containing class folders"""
    for root, dirs, files in os.walk(base_path):
        # Check if this directory contains subdirectories with images
        if dirs and any(len(os.listdir(os.path.join(root, d))) > 0 for d in dirs if os.path.isdir(os.path.join(root, d))):
            # Check if subdirectories contain image files
            for d in dirs:
                dir_path = os.path.join(root, d)
                files_in_dir = os.listdir(dir_path)
                if any(f.lower().endswith(('.png', '.jpg', '.jpeg')) for f in files_in_dir):
                    return root
    return base_path

data_root = find_data_root(extract_path)
print(f"Data root directory: {data_root}")

# Get class folders
class_folders = [d for d in os.listdir(data_root) 
                 if os.path.isdir(os.path.join(data_root, d)) and not d.startswith('.')]
class_folders.sort()

print(f"\nFound {len(class_folders)} classes: {class_folders}")

# Count images per class
for class_name in class_folders:
    class_path = os.path.join(data_root, class_name)
    num_images = len([f for f in os.listdir(class_path) 
                     if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    print(f"  {class_name}: {num_images} images")

## Step 4: Create Train/Test Split

We'll move a percentage of images from each class to a test folder

In [ ]:
# Configuration
TEST_SPLIT_RATIO = 0.2  # 20% for testing, 80% for training
RANDOM_SEED = 42

# Create train and test directories
train_dir = '/content/train'
test_dir = '/content/test'

# Remove if already exists
if os.path.exists(train_dir):
    shutil.rmtree(train_dir)
if os.path.exists(test_dir):
    shutil.rmtree(test_dir)

os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

print("Created directories:")
print(f"  Train: {train_dir}")
print(f"  Test: {test_dir}")

In [ ]:
# Split data into train and test
random.seed(RANDOM_SEED)

split_summary = {}

for class_name in class_folders:
    # Create class folders in train and test
    train_class_dir = os.path.join(train_dir, class_name)
    test_class_dir = os.path.join(test_dir, class_name)
    
    os.makedirs(train_class_dir, exist_ok=True)
    os.makedirs(test_class_dir, exist_ok=True)
    
    # Get all images from this class
    source_class_dir = os.path.join(data_root, class_name)
    all_images = [f for f in os.listdir(source_class_dir) 
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    # Shuffle images
    random.shuffle(all_images)
    
    # Calculate split point
    num_test = max(1, int(len(all_images) * TEST_SPLIT_RATIO))
    num_train = len(all_images) - num_test
    
    test_images = all_images[:num_test]
    train_images = all_images[num_test:]
    
    # Copy images to respective folders
    for img in test_images:
        src = os.path.join(source_class_dir, img)
        dst = os.path.join(test_class_dir, img)
        shutil.copy2(src, dst)
    
    for img in train_images:
        src = os.path.join(source_class_dir, img)
        dst = os.path.join(train_class_dir, img)
        shutil.copy2(src, dst)
    
    split_summary[class_name] = {
        'total': len(all_images),
        'train': num_train,
        'test': num_test
    }
    
    print(f"{class_name}: {len(all_images)} total → Train: {num_train}, Test: {num_test}")

print("\n✓ Train/Test split completed successfully!")

In [ ]:
# Visualize split distribution
classes = list(split_summary.keys())
train_counts = [split_summary[c]['train'] for c in classes]
test_counts = [split_summary[c]['test'] for c in classes]

x = np.arange(len(classes))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, train_counts, width, label='Train', alpha=0.8)
bars2 = ax.bar(x + width/2, test_counts, width, label='Test', alpha=0.8)

ax.set_xlabel('Dollar Bill Class', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('Train/Test Split Distribution', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

total_train = sum(train_counts)
total_test = sum(test_counts)
print(f"\nTotal Training Images: {total_train}")
print(f"Total Testing Images: {total_test}")
print(f"Total Images: {total_train + total_test}")

## Step 5: Visualize Sample Images

In [ ]:
# Display sample images from each class
fig, axes = plt.subplots(len(class_folders), 3, figsize=(12, 4*len(class_folders)))

if len(class_folders) == 1:
    axes = axes.reshape(1, -1)

for i, class_name in enumerate(class_folders):
    class_path = os.path.join(train_dir, class_name)
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    for j in range(min(3, len(images))):
        img_path = os.path.join(class_path, images[j])
        img = Image.open(img_path)
        axes[i, j].imshow(img)
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(f'{class_name}\n{images[j]}', fontsize=10, fontweight='bold')
        else:
            axes[i, j].set_title(images[j], fontsize=9)

plt.suptitle('Sample Dollar Bill Images from Each Class', fontsize=16, fontweight='bold', y=1.0)
plt.tight_layout()
plt.show()

## Step 6: Data Preprocessing and Augmentation

In [ ]:
# Image parameters
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
NUM_CLASSES = len(class_folders)

print(f"Image size: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Only rescaling for test data (no augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

# Create data generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\nTraining samples: {train_generator.samples}")
print(f"Testing samples: {test_generator.samples}")
print(f"\nClass indices: {train_generator.class_indices}")

## Step 7: Build CNN Model

We'll use **EfficientNetB0** with transfer learning for better performance

In [ ]:
def create_model(num_classes, model_type='efficientnet'):
    """
    Create a CNN model for dollar bill classification
    
    Args:
        num_classes: Number of output classes
        model_type: 'efficientnet', 'mobilenet', or 'custom'
    """
    
    if model_type == 'efficientnet':
        # EfficientNetB0 with transfer learning
        base_model = EfficientNetB0(
            include_top=False,
            weights='imagenet',
            input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
        )
        base_model.trainable = False  # Freeze base model initially
        
        model = models.Sequential([
            base_model,
            layers.GlobalAveragePooling2D(),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            layers.Dense(256, activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            layers.Dense(num_classes, activation='softmax')
        ])
        
    elif model_type == 'mobilenet':
        # MobileNetV2 - lighter model
        base_model = MobileNetV2(
            include_top=False,
            weights='imagenet',
            input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
        )
        base_model.trainable = False
        
        model = models.Sequential([
            base_model,
            layers.GlobalAveragePooling2D(),
            layers.Dropout(0.3),
            layers.Dense(128, activation='relu'),
            layers.Dropout(0.2),
            layers.Dense(num_classes, activation='softmax')
        ])
        
    else:
        # Custom CNN from scratch
        model = models.Sequential([
            # Block 1
            layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            
            # Block 2
            layers.Conv2D(64, (3, 3), activation='relu'),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            
            # Block 3
            layers.Conv2D(128, (3, 3), activation='relu'),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            
            # Block 4
            layers.Conv2D(256, (3, 3), activation='relu'),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            
            # Dense layers
            layers.Flatten(),
            layers.Dense(512, activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(256, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(num_classes, activation='softmax')
        ])
    
    return model

# Create model
model = create_model(NUM_CLASSES, model_type='efficientnet')

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 8: Setup Training Callbacks

In [ ]:
# Callbacks for better training
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_dollar_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("Callbacks configured:")
print("  - Early Stopping (patience=10)")
print("  - Learning Rate Reduction (factor=0.5, patience=5)")
print("  - Model Checkpoint (save best model)")

## Step 9: Train the Model

In [ ]:
# Training parameters
EPOCHS = 50

print(f"Starting training for {EPOCHS} epochs...\n")

# Train model
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=test_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Training completed!")

## Step 10: Visualize Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]

print(f"\nFinal Training Accuracy: {final_train_acc:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")
print(f"Final Training Loss: {final_train_loss:.4f}")
print(f"Final Validation Loss: {final_val_loss:.4f}")

## Step 11: Evaluate on Test Data

In [ ]:
# Load best model
model = keras.models.load_model('best_dollar_model.h5')
print("Loaded best model from checkpoint\n")

# Evaluate on test set
test_loss, test_accuracy = model.evaluate(test_generator)

print(f"\n{'='*50}")
print(f"TEST SET PERFORMANCE")
print(f"{'='*50}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")
print(f"{'='*50}")

## Step 12: Generate Predictions and Confusion Matrix

In [ ]:
# Get predictions
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)

# Get true labels
true_classes = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

print(f"\nGenerated predictions for {len(predicted_classes)} test images")

In [ ]:
# Classification Report
print("\nCLASSIFICATION REPORT")
print("="*70)
print(classification_report(true_classes, predicted_classes, target_names=class_labels))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_labels, 
            yticklabels=class_labels,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Dollar Bill Classification', fontsize=14, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Per-class accuracy
print("\nPER-CLASS ACCURACY")
print("="*40)
for i, class_name in enumerate(class_labels):
    class_correct = cm[i, i]
    class_total = cm[i].sum()
    class_acc = (class_correct / class_total * 100) if class_total > 0 else 0
    print(f"{class_name:15s}: {class_acc:6.2f}% ({class_correct}/{class_total})")

## Step 13: Visualize Predictions

In [ ]:
# Visualize sample predictions
test_generator.reset()
x_batch, y_batch = next(test_generator)
predictions_batch = model.predict(x_batch)

# Show 12 random predictions
num_samples = min(12, len(x_batch))
indices = random.sample(range(len(x_batch)), num_samples)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for idx, i in enumerate(indices):
    true_label = class_labels[np.argmax(y_batch[i])]
    pred_label = class_labels[np.argmax(predictions_batch[i])]
    confidence = np.max(predictions_batch[i]) * 100
    
    axes[idx].imshow(x_batch[i])
    axes[idx].axis('off')
    
    color = 'green' if true_label == pred_label else 'red'
    title = f'True: {true_label}\nPred: {pred_label}\nConf: {confidence:.1f}%'
    axes[idx].set_title(title, fontsize=10, color=color, fontweight='bold')

plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## Step 14: Analyze Misclassifications

In [ ]:
# Find misclassified examples
misclassified_indices = np.where(predicted_classes != true_classes)[0]
num_misclassified = len(misclassified_indices)

print(f"Total Misclassifications: {num_misclassified}/{len(true_classes)}")
print(f"Accuracy: {(1 - num_misclassified/len(true_classes))*100:.2f}%")

if num_misclassified > 0:
    print(f"\nShowing up to 8 misclassified examples...\n")
    
    # Show some misclassified examples
    num_show = min(8, num_misclassified)
    show_indices = random.sample(list(misclassified_indices), num_show)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    test_generator.reset()
    all_images = []
    all_labels = []
    
    for x, y in test_generator:
        all_images.extend(x)
        all_labels.extend(y)
        if len(all_images) >= test_generator.samples:
            break
    
    for idx, i in enumerate(show_indices):
        true_label = class_labels[true_classes[i]]
        pred_label = class_labels[predicted_classes[i]]
        confidence = np.max(predictions[i]) * 100
        
        axes[idx].imshow(all_images[i])
        axes[idx].axis('off')
        axes[idx].set_title(f'True: {true_label}\nPredicted: {pred_label}\nConf: {confidence:.1f}%',
                          fontsize=10, color='red', fontweight='bold')
    
    plt.suptitle('Misclassified Examples', fontsize=16, fontweight='bold', color='red')
    plt.tight_layout()
    plt.show()
else:
    print("\n🎉 Perfect classification! No misclassifications found.")

## Step 15: Save Model and Results

In [ ]:
# Save final model
model.save('dollar_bill_classifier_final.h5')
print("✓ Model saved as 'dollar_bill_classifier_final.h5'")

# Save model architecture as JSON
model_json = model.to_json()
with open('model_architecture.json', 'w') as json_file:
    json_file.write(model_json)
print("✓ Model architecture saved as 'model_architecture.json'")

# Save training history
with open('training_history.json', 'w') as f:
    json.dump(history.history, f)
print("✓ Training history saved as 'training_history.json'")

# Save class indices
with open('class_indices.json', 'w') as f:
    json.dump(train_generator.class_indices, f)
print("✓ Class indices saved as 'class_indices.json'")

# Save results summary
results_summary = {
    'test_accuracy': float(test_accuracy),
    'test_loss': float(test_loss),
    'num_classes': NUM_CLASSES,
    'class_labels': class_labels,
    'total_train_images': train_generator.samples,
    'total_test_images': test_generator.samples,
    'num_misclassified': int(num_misclassified)
}

with open('results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=4)
print("✓ Results summary saved as 'results_summary.json'")

print("\n" + "="*50)
print("All files saved successfully!")
print("="*50)

## Step 16: Download Results (Optional)

In [ ]:
# Download all result files
from google.colab import files

print("Downloading result files...\n")

files_to_download = [
    'best_dollar_model.h5',
    'dollar_bill_classifier_final.h5',
    'model_architecture.json',
    'training_history.json',
    'class_indices.json',
    'results_summary.json'
]

for filename in files_to_download:
    if os.path.exists(filename):
        files.download(filename)
        print(f"✓ Downloaded: {filename}")
    else:
        print(f"✗ Not found: {filename}")

print("\nDownload complete!")

## 📊 Final Summary

In [ ]:
print("\n" + "="*60)
print("QUIZ 2: DOLLAR BILL DETECTION - FINAL SUMMARY")
print("="*60)
print(f"\n📁 Dataset:")
print(f"   - Total Classes: {NUM_CLASSES}")
print(f"   - Class Labels: {', '.join(class_labels)}")
print(f"   - Training Images: {train_generator.samples}")
print(f"   - Testing Images: {test_generator.samples}")
print(f"\n🤖 Model:")
print(f"   - Architecture: EfficientNetB0 + Transfer Learning")
print(f"   - Input Size: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"   - Total Parameters: {model.count_params():,}")
print(f"\n📈 Performance:")
print(f"   - Test Accuracy: {test_accuracy*100:.2f}%")
print(f"   - Test Loss: {test_loss:.4f}")
print(f"   - Misclassified: {num_misclassified}/{len(true_classes)}")
print(f"\n✅ Requirements Met:")
print(f"   ✓ Created separate test folder")
print(f"   ✓ Moved images to test folder (not in training)")
print(f"   ✓ Trained CNN model")
print(f"   ✓ Tested on separate test data")
print(f"   ✓ Calculated accuracy metrics")
print("\n" + "="*60)
print("🎉 QUIZ 2 COMPLETED SUCCESSFULLY!")
print("="*60)